[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Occupation_Classification.ipynb)


# Occupational Classification from Survey Evidence

## Data analysis and classification methodology

I approach occupational classification as an inference from the work a respondent actually performs. I distinguish the information supplied by the respondent, the consistency of the occupational descriptions, the specificity of the recorded classification and the ability of a model to generalize. I retrieve the original workbook from the project repository and retain its source values throughout the analysis.

In [ ]:
import io, hashlib, urllib.request, re, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings('ignore',category=UserWarning)
SEED=2026
URL='https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx'
SHA='e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac'
raw=urllib.request.urlopen(URL,timeout=90).read()
assert hashlib.sha256(raw).hexdigest()==SHA, 'The workbook does not match the source checksum'
book=pd.ExcelFile(io.BytesIO(raw)); print('Workbook sheets:',book.sheet_names, 'Source bytes:',len(raw))

In [ ]:
data=pd.read_excel(io.BytesIO(raw),sheet_name='Survey Dataset',usecols='A:R',dtype=str)
metadata=pd.read_excel(io.BytesIO(raw),sheet_name='Metadata',usecols='A:D',dtype=str)
data.columns=data.columns.astype(str).str.strip(); metadata.columns=metadata.columns.astype(str).str.strip()
data=data.loc[data['ID'].notna()].copy().reset_index(drop=True)
assert len(data)==200 and data['ID'].nunique()==200 and 'SemelMishlachSofi' in data
assert len(data.columns)==18
original=data.copy(deep=True)
print('Survey dimensions:',data.shape,'Metadata dimensions:',metadata.shape)
display(metadata.head(20));display(data.head(3))

## Distinguishing sources of uncertainty

I distinguish three situations that have different implications for classification. A respondent may leave a question unanswered. The descriptions supplied across questions may disagree. A professional coder may assign only part of the occupational code. I examine these situations separately because a single missing value indicator would obscure their different meanings.

In [ ]:
TEXT=['YeshuvAvoda','ShemAvoda','SugAvoda','ShemMachlaka','SugMachlaka','EzoAvoda','TeurPeula','TeurTafkid']
OCC=['SugAvoda','ShemMachlaka','SugMachlaka','EzoAvoda','TeurPeula','TeurTafkid']
TARGET='SemelMishlachSofi'
def clean(x):
    if pd.isna(x):return ''
    return re.sub(r'\s+',' ',str(x).strip()).casefold()
def meaningful(x):return bool(clean(x)) and clean(x) not in {'כנ"ל','כנ״ל','כנל','חסוי','לא יכול לפרט'}
for c in TEXT:data[c+'_observed']=data[c].map(meaningful).astype(int)
data['occupation_evidence_count']=data[[c+'_observed' for c in OCC]].sum(axis=1)
data['target_resolution']=data[TARGET].fillna('').map(lambda x:sum(ch!='X' for ch in str(x)) if len(str(x))==4 else 0)
data['target_complete']=data[TARGET].fillna('').str.fullmatch(r'\d{4}').fillna(False)
display(data.groupby('occupation_evidence_count',dropna=False).agg(records=('ID','size'),fully_coded=('target_complete','sum'),mean_code_resolution=('target_resolution','mean')))
print('Records with incomplete occupational codes:',int((~data.target_complete).sum()))

## Administrative identity and occupational evidence

I treat `ID` as the record identifier rather than an occupational attribute. I compare substantive descriptions across records to identify repetition and conflicting classifications. A repeated description does not establish that two records belong to the same respondent. I therefore use occupational evidence signatures for analysis and grouping without using the record identifier as a predictor.

In [ ]:
def signature(row,cols):return ' | '.join(clean(row[c]) for c in cols)
data['evidence_signature']=data.apply(lambda r:signature(r,OCC),axis=1)
data['all_text_signature']=data.apply(lambda r:signature(r,TEXT),axis=1)
rep=data.groupby('evidence_signature').agg(n=('ID','size'),targets=(TARGET,lambda s:sorted(set(s.dropna()))),target_count=(TARGET,'nunique')).sort_values('n',ascending=False)
print('Description signatures shared by multiple records:',int((rep.n>1).sum()))
print('Description signatures associated with different occupational codes:',int(((rep.n>1)&(rep.target_count>1)).sum()))
display(rep[(rep.n>1)&(rep.target_count>1)].head(12))

## Distinguishing occupational duties from workplace context

I separate descriptions of the respondent's duties from information about the employer and workplace. The same organization can employ people in different occupations, while similar duties can occur across different organizations. I measure the contribution of each information group through feature ablation rather than assuming that either group is sufficient on its own.

In [ ]:
ROLE=['EzoAvoda','TeurPeula','TeurTafkid','MenahelEtMi']
CONTEXT=['ShemAvoda','SugAvoda','ShemMachlaka','SugMachlaka','YeshuvAvoda']
OTHER=['MaamadAvoda','MakorSachar','TeudaGvoha','shnotlimud','Gil','TarichHathala']
def pack(frame,cols):
    return frame.apply(lambda r:' '.join(f'{c} {clean(r[c])}' for c in cols if meaningful(r[c])),axis=1)
for name,cols in [('role',ROLE),('context',CONTEXT),('combined',ROLE+CONTEXT+OTHER)]:data['text_'+name]=pack(data,cols)
valid=data.loc[data.target_complete].copy()
print('Records with complete occupational codes:',len(valid),'Distinct complete occupational codes:',valid[TARGET].nunique())
print('Most frequent complete occupational codes:');display(valid[TARGET].value_counts().head(12))

## Evaluation across distinct occupational descriptions

I group records by their substantive occupational descriptions before cross validation so that identical descriptions do not appear in both training and test partitions. I compare a majority class baseline with independently trained text classifiers. I report held out performance together with the limitations created by rare occupational codes in this simulated sample.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
models={
 'Majority class baseline':DummyClassifier(strategy='most_frequent'),
 'Word TFIDF logistic regression':make_pipeline(TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1),LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED)),
 'Character TFIDF support vector classifier':make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,5),min_df=1,max_features=25000),LinearSVC(class_weight='balanced',random_state=SEED)),
 'Character TFIDF naive Bayes':make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,4),min_df=1,max_features=25000),ComplementNB(alpha=0.5))}
groups=valid['evidence_signature']; folds=min(5,groups.nunique())
assert folds>=2
cv=GroupKFold(n_splits=folds)
results=[]; predictions=[]
for view in ['role','context','combined']:
    X=valid['text_'+view].reset_index(drop=True);y=valid[TARGET].reset_index(drop=True);g=groups.reset_index(drop=True)
    for name,model in models.items():
        yp=np.empty(len(y),dtype=object);unseen=[]
        for train,test in cv.split(X,y,g):
            fitted=clone(model);fitted.fit(X.iloc[train],y.iloc[train]);yp[test]=fitted.predict(X.iloc[test]);unseen.extend([int(i) for i in test if y.iloc[i] not in set(y.iloc[train])])
        score={'view':view,'model':name,'accuracy':accuracy_score(y,yp),'macro_f1':f1_score(y,yp,average='macro',zero_division=0),'unseen_label_rows':len(unseen)}
        results.append(score)
        predictions.append(pd.DataFrame({'ID':valid.ID.to_numpy(),'view':view,'model':name,'actual':y,'predicted':yp,'correct':y.to_numpy()==yp}))
results=pd.DataFrame(results).sort_values(['macro_f1','accuracy'],ascending=False)
display(results)
print('Highest scoring evaluated configuration:',results.iloc[0].to_dict())

## Review of classification decisions

I treat each prediction as a proposed classification. I prioritize records with conflicting occupational evidence, incomplete source classifications or insufficient substantive descriptions for professional review. I retain the original responses and the reasons for each review decision.

In [ ]:
best=results.iloc[0];best_preds=next(p for p in predictions if p['view'].iloc[0]==best['view'] and p['model'].iloc[0]==best['model'])
review=data[['ID',TARGET,'occupation_evidence_count','target_resolution','evidence_signature']].copy()
review['conflicting_evidence']=review['evidence_signature'].map(rep.target_count).gt(1)
review['needs_human_review']=(~data.target_complete)|(data.occupation_evidence_count<2)|review.conflicting_evidence
review=review.merge(best_preds[['ID','predicted','correct']],on='ID',how='left')
display(review.sort_values(['needs_human_review','occupation_evidence_count'],ascending=[False,True]).head(25))
print('Records selected for professional review:',int(review.needs_human_review.sum()),'of',len(review))
assert original.shape==(200,18) and original.equals(data[original.columns])
print('I confirmed that all 200 source records and 18 source variables remain unchanged')

In [ ]:
output=Path('cbs_third_generation_results');output.mkdir(exist_ok=True)
results.to_csv(output/'model_ablation.csv',index=False)
review.to_csv(output/'human_review_queue.csv',index=False,encoding='utf-8-sig')
pd.concat(predictions,ignore_index=True).to_csv(output/'held_out_predictions.csv',index=False)
manifest={'source_sha256':SHA,'rows':len(original),'columns':len(original.columns),'full_label_rows':len(valid),'cv_folds':folds,'best':best.to_dict()}
(output/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('I saved the analysis outputs:',[p.name for p in output.iterdir()])

## Conclusions

1. **Record identity.** I use `ID` to identify records and examine repeated occupational descriptions separately.

2. **Information quality.** I distinguish unanswered questions, uninformative descriptions, conflicting evidence and incomplete classifications.

3. **Feature relevance.** I measure how descriptions of actual duties contribute relative to organizational context.

4. **Generalization.** I evaluate predictions on separated description groups and interpret performance in the context of a small simulated sample.

5. **Operational use.** I retain source responses, record model decisions and direct cases with insufficient evidence to professional review.

The executable analysis calculates the reported numerical results.